In [ ]:
!pip install unsloth trl datasets

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

# 1. Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-3B-bnb-4bit", # Excellent choice for code generation
    max_seq_length = 4096, # Increased to handle long HTML outputs
    dtype = None,
    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 3. Load your custom JSON dataset
# This replaces the Hugging Face LaTeX_OCR dataset
dataset = load_dataset("json", data_files="game_dataset.json", split="train")

In [ ]:
# 4. Format prompts using the chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

<IPython.core.display.Math object>

In [ ]:
def formatting_prompts_func(examples):
    convs = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convs]
    return { "text" : texts }

converted_dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Train the model
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    train_dataset = converted_dataset,
    dataset_text_field = "text",
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Change to num_train_epochs = 3 for a full training cycle
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

 $$H ^ { \prime } = \beta N \int d \lambda \left\{ \frac { 1 } { 2 \beta ^ { 2 } N ^ { 2 } } \partial _ { \lambda } \zeta ^ { \dagger } \partial _ { \lambda } \zeta + V ( \lambda ) \zeta ^ { \dagger } \zeta \right\} .$$<|im_end|>
<|endoftext|>


In [ ]:
# 6. Save the model locally
model.save_pretrained("qwen_game_generator_lora")
tokenizer.save_pretrained("qwen_game_generator_lora")

Unsloth: Model does not have a default image size - using 512


In [ ]:
# 7. Test Inference (Text-to-Text)
FastLanguageModel.for_inference(model)

# Test it with a brand new prompt
messages = [
    {"role": "system", "content": "You are an expert game development AI. The user will provide a short game concept. You must generate a complete, playable, single-file HTML game and return it strictly as a JSON object containing the keys: title, prompt, how_to, controls, code, category, tagline, thumbnail_url, and soundtrack_prompt."},
    {"role": "user", "content": "a platformer game about a jumping frog trying to catch flies"}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = tokenizer([input_text], return_tensors="pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Note: max_new_tokens is set very high here to ensure the full HTML block prints
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=4096,
    use_cache=True,
    temperature=0.7
)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,686 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 23,110,656 of 2,236,352,320 (1.03% trained)


Step,Training Loss
1,0.631942
2,0.781121
3,0.835430
4,0.692773
5,0.668855
6,0.895739
7,0.856702
8,0.714944
9,0.512467
10,0.560102
